# Model Selection Criteria

**Module:** 06 — LLM Models

A practical decision rubric and selection pattern for real products—not leaderboard chasing.


## How to Use This Notebook

Read each section as a mini-lesson, run every code cell, then change inputs to stress-test your intuition. API examples use placeholders such as `YOUR_API_KEY` or `os.getenv(...)` — never hard-code secrets.

Each major topic includes: definition, why it matters, how it works, intuition, pitfalls, when-to-use guidance, practical demos, and a short exercise.


## Learning Objectives

By the end of this notebook, you will be able to:

- Apply a multi-factor decision rubric
- Run a practical selection pattern with bakeoffs
- Document router policies for production
- Revisit choices when prices/SKUs change


## Decision Rubric

**Definition.** Score candidates on quality (your evals), latency, cost, context, tools/multimodal needs, license/compliance, vendor lock-in, and ops complexity.

**Why it matters.** Prevents buying the 'smartest' model that fails residency or burns budget.

**How it works.** Weight factors for the use case; eliminate deal-breakers first; then bake off finalists.

**Intuition.** Buying a vehicle: terrain and cargo first, top speed later.

**Common pitfalls.**
- Single-metric selection (only MMLU)
- Ignoring p95 latency

**When to use.** Every net-new GenAI feature.

| Factor | Questions |
|--------|-----------|
| Quality | Pass on *our* eval? |
| Latency | p95 OK? |
| Cost | $/1k at traffic? |
| Compliance | Region/license? |
| Features | Tools/vision? |
| Ops | Self-host burden? |


In [ ]:
# Demo 1 — weighted rubric
candidates = {
  "mini": {"quality": 0.72, "latency": 0.9, "cost": 0.95, "compliance": 0.8},
  "flagship": {"quality": 0.9, "latency": 0.6, "cost": 0.4, "compliance": 0.85},
}
weights = {"quality": 0.4, "latency": 0.2, "cost": 0.25, "compliance": 0.15}

def score(row):
    return sum(row[k]*w for k,w in weights.items())
for name, row in candidates.items():
    print(name, round(score(row), 3))


In [ ]:
# Demo 2 — deal-breakers first
def eligible(row):
    return row.get("residency_ok") and row.get("license_ok")
print(eligible({"residency_ok": True, "license_ok": False}))


In [ ]:
# Demo 3 — context requirement
need_ctx = 100_000
print({"gemini_or_long_ctx": need_ctx >= 100_000, "rag_instead": True})


In [ ]:
# Demo 4 — tool requirement
print({"needs_tools": True, "filter": "candidates_with_stable_tool_calling"})


### Try it yourself — Decision Rubric

1. Weight a rubric for an internal HR bot vs a consumer creative writer.
2. List your organization's top three deal-breakers.


## Practical Pattern

**Definition.** A repeatable selection pattern: constrain → shortlist → harness bakeoff → shadow traffic → router policy → revisit quarterly.

**Why it matters.** Turns model choice into an engineering process instead of a debate club.

**How it works.** Automate evals; keep a champion/challenger; document routers.

**Intuition.** Sports tryouts with stats, then game-time substitutions.

**Common pitfalls.**
- One-off bakeoffs never refreshed
- No shadow mode before cutover

**When to use.** Platform teams and product launches.

```mermaid
flowchart TB
  C[Constrain] --> S[Shortlist]
  S --> B[Bakeoff]
  B --> H[Shadow]
  H --> R[Router policy]
  R --> Q[Quarterly revisit]
```


In [ ]:
# Demo 1 — selection pipeline
steps = ["constrain", "shortlist3", "offline_bakeoff", "shadow", "cutover", "quarterly_review"]
print(" → ".join(steps))


In [ ]:
# Demo 2 — champion/challenger
state = {"champion": "gpt-4.1-mini", "challenger": "claude-sonnet-class", "shadow_pct": 10}
print(state)


In [ ]:
# Demo 3 — router policy
def route(task, length_tokens):
    if task == "embed":
        return "embeddings"
    if length_tokens > 50_000:
        return "long-context-family"
    if task == "code_repair":
        return "coding-strong-model"
    return "default-mini"
print(route("code_repair", 1000), route("faq", 800))


In [ ]:
# Demo 4 — quarterly trigger
triggers = ["price change >20%", "new SKU", "eval drift", "incident"]
print(triggers)


### Try it yourself — Practical Pattern

1. Write a router policy for your app in pseudocode.
2. Schedule a quarterly review agenda with 5 items.


## Glossary

- **champion/challenger**: Current prod model vs candidate
- **shadow traffic**: Send copies of live requests to a candidate


### Workshop drill — Model Selection Criteria (1)

Fill a comparison row: strengths, limits, typical SKU/API name, and one eval signal.


In [ ]:
# Workshop drill 1 — Model Selection Criteria
import json
print(json.dumps({
  'family': 'Model Selection Criteria',
  'strengths': ['...'],
  'limits': ['...'],
  'api_name': '...',
  'eval_signal': '...'
}, indent=2))


### Workshop drill — Model Selection Criteria (2)

Draft a realistic request JSON with YOUR_API_KEY placeholder (do not call the network).


In [ ]:
# Workshop drill 2 — Model Selection Criteria
import json
YOUR_API_KEY='YOUR_API_KEY'
req={'model':'MODEL_ID','messages':[{'role':'user','content':'Hello'}]}
print(json.dumps(req, indent=2))
print('Authorization: Bearer', YOUR_API_KEY[:8]+'...')


### Workshop drill — Model Selection Criteria (3)

Write a go/no-go for this family on a regulated enterprise FAQ bot.


In [ ]:
# Workshop drill 3 — Model Selection Criteria
checks=['license','data_residency','tool_calling','vision_needed','cost']
for c in checks:
    print(f'[ ] {c}: pass/fail because ...')


### Workshop drill — Model Selection Criteria (4)

List three prompts that stress this family's claimed strengths.


In [ ]:
# Workshop drill 4 — Model Selection Criteria
for i,p in enumerate(['prompt1','prompt2','prompt3'],1):
    print(i, p)


### Workshop drill — Model Selection Criteria (5)

Cost sketch: estimate monthly $ for 10M input + 2M output tokens.


In [ ]:
# Workshop drill 5 — Model Selection Criteria
in_tok, out_tok = 10_000_000, 2_000_000
in_price = out_price = 1.0  # $/1M tok placeholders — replace with card prices
print('approx $', in_tok/1e6*in_price + out_tok/1e6*out_price)


### Workshop drill — Model Selection Criteria (6)

Router rule: when would you escalate from a small model in this family to a larger one?


In [ ]:
# Workshop drill 6 — Model Selection Criteria
def route(complexity: float) -> str:
    return 'large' if complexity > 0.7 else 'small'
for c in [0.2, 0.7, 0.95]:
    print(c, route(c))


### Workshop drill — Model Selection Criteria (7)

Write a failure-mode card: hallucination, latency, license, and vendor lock-in.


In [ ]:
# Workshop drill 7 — Model Selection Criteria
for risk in ['hallucination','latency','license','lock_in']:
    print(f'{risk}: mitigation=...')


### Workshop drill — Model Selection Criteria (8)

Invent a shadow-traffic plan: % of live traffic, success metrics, rollback.


In [ ]:
# Workshop drill 8 — Model Selection Criteria
plan = {'shadow_pct': 5, 'metrics': ['latency_p95','task_pass'], 'rollback': 'champion'}
print(plan)


## Summary & Key Takeaways

- Use a weighted rubric after deal-breakers
- Bake off on your harness, then shadow
- Routers beat one-model-forever thinking
- Revisit when prices, SKUs, or evals move

### Practice

Produce a one-page model selection brief for your next feature.


## Self-Check

1. Can you explain the main idea of each section in one sentence?
2. Which technique would you use first in production, and why?
3. What failure mode should you monitor after shipping?
4. What metric would tell you the system got worse?


In [ ]:
checklist = [
    "I can restate the learning objectives",
    "I ran/adapted at least two code examples",
    "I know which env vars/keys this topic needs",
    "I noted one risk (cost, safety, latency, or quality)",
    "I can name one pitfall and its mitigation",
]
for i, item in enumerate(checklist, 1):
    print(f"{i}. [ ] {item}")
